# Stage 6: efficiency benchmark

Results of `python -m src.benchmark`. Everything is measured on the same machine (specs below).

- **train time**: final fit on train + validation;
- **model-only latency**: one sample, and per sample in a batch of 1,000 (median, after warm-up);
- **end-to-end latency**: building the features for one forecast + the model;
- **size**: the model file on disk;
- **exports**: ONNX (float32) for the GBMs and the LSTM, int8 dynamic quantization for the LSTM,
  each scored again on the test set.

In [ ]:
import pandas as pd
from IPython.display import Image

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
TABLES = "../results/tables/"
FIGURES = "../results/figures/"

In [ ]:
eff = pd.read_csv(TABLES + "06_efficiency.csv")
eff[["cpu", "cores", "ram_gb", "os", "python"]].drop_duplicates()

## 1. Cost and accuracy of every model

In [ ]:
columns = ["horizon", "model", "train_seconds", "single_ms", "batch_ms_per_sample", "end_to_end_single_ms", "size_kB", "test_nmae_pct"]
eff[columns].round(4)

## 2. Feature construction dominates one forecast
`feature_ms` is the same for every model of a horizon.

In [ ]:
eff.groupby("horizon")["feature_ms"].first().round(1)

## 3. Exports: does accuracy change?
A change of 0 means the exported model gives the same error as the original.

In [ ]:
exports = eff[eff["model"].str.contains("onnx|int8")]
exports[["horizon", "model", "size_kB", "single_ms", "test_nmae_pct", "nmae_change_vs_original_pct_points", "note"]].round(4)

## 4. Accuracy vs latency

In [ ]:
Image(FIGURES + "06_accuracy_vs_latency.png")